# Pixi-based Dream3DNX installation

As of 5/10/2026 `dream3dnx` requires python 3.12.

Follow either of the options below to install dream3dnx using pixi or follow the dream3dnx conda/mamba instructions in their [python docs](https://www.dream3d.io/python_docs/Installation.html).

## Option 1: Run Commands
```
pixi project channel add bluequartzsoftware
pixi add dream3dnx
pixi install
```
## Option 2: Update pyproject.toml
Add the following lines to the `pyproject.toml`
```toml
[tool.pixi.workspace]
channels = ["conda-forge", "bluequartzsoftware"]
platforms = ["win-64"]

[tool.pixi.pypi-dependencies]
misalign = { path = ".", editable = true }

[tool.pixi.environments]
default = { solve-group = "default" }
dev = { features = ["dev"], solve-group = "default" }
example-hdf5 = { features = ["example-hdf5"], solve-group = "default" }
ipy = { features = ["ipy"], solve-group = "default" }
relation-visualize = { features = ["relation-visualize"], solve-group = "default" }
all = { features = ["dev","example-hdf5","ipy","relation-visualize"], solve-group = "default" }

[tool.pixi.tasks]

[tool.pixi.feature.example-hdf5.dependencies]
dream3dnx = ">=26.3.23,<27"

[tool.pixi.dependencies]
dream3dnx = ">=26.3.23,<27"

[tool.ty.environment]
extra-paths = [".pixi/envs/all/Lib/site-packages"]
```
And then run `pixi install -e all`

# Create `.dream3d` from set of images

In [151]:
from pathlib import Path
import logging

In [152]:
import simplnx as nx
import itkimageprocessing as cxitk
# import orientationanalysis as cxor

In [153]:
output_dir = Path(".")
data_structure = nx.DataStructure()

In [154]:

# logging.getLogger().setLevel(logging.INFO) # Optional - set logger level to show info logs
def log_filter_result(result:nx.IFilter.ExecuteResult,filter:nx.IFilter|None=None,):
    """
    This function will log any warnings or errors from the `result`.
    It logs warnings and errors at their respective log levels.
    It logs a succesful result with no warnings or errors at the `info` level.
    If uncommented it throws a `RuntimeError` if any errors are present.
    Either modify this function or handle the error in a `try:` `except:` statement.
    """
    if filter is None:
        filter_name="Filter"
    else:
        filter_name=filter.name()
    
    # logging.info(msg=f" {filter_name} : {result}") # Optional - info log the full result
    for warning in result.warnings:
        logging.warning(msg=f" {filter_name} : {warning.code} {warning.code}")
    for error in result.errors:
        logging.error(msg=f" {filter_name} : {error}")
    # if result.errors: # Optional - raise a RuntimeError if any errors are present
    #     raise RuntimeError(result)
    if not result.warnings and not result.errors:
        logging.info(msg=f" {filter_name} : No Warnings or Errors")


In [155]:
for i in range(10):
    result=cxitk.ITKImageReaderFilter.execute(data_structure=data_structure,
        file_name=Path(f"../project_a/image_a{i+1:02d}.jpg"),
        change_origin=True,
        origin=[0,1100*i,0],
        output_geometry_path=nx.DataPath(f"image_a{i+1:02d}.jpg"))
    log_filter_result(result=result)
print(data_structure.hierarchy_to_str())

|--image_a01.jpg
  |--Cell Data
    |--ImageData
|--image_a02.jpg
  |--Cell Data
    |--ImageData
|--image_a03.jpg
  |--Cell Data
    |--ImageData
|--image_a04.jpg
  |--Cell Data
    |--ImageData
|--image_a05.jpg
  |--Cell Data
    |--ImageData
|--image_a06.jpg
  |--Cell Data
    |--ImageData
|--image_a07.jpg
  |--Cell Data
    |--ImageData
|--image_a08.jpg
  |--Cell Data
    |--ImageData
|--image_a09.jpg
  |--Cell Data
    |--ImageData
|--image_a10.jpg
  |--Cell Data
    |--ImageData



In [ ]:
# img:nx.UInt8Array=data_structure['image_a04.jpg/Cell Data/ImageData']
# plt.imshow(img.npview()[0])


In [156]:
# Use the WriteDREAM3DFilter to write out the modified DataStructure to disk
result = nx.WriteDREAM3DFilter.execute(data_structure=data_structure,
                                    export_file_path=output_dir.joinpath("project_a.dream3d"),
                                    write_xdmf_file=False)
log_filter_result(result)